# Tutorial 02 — W7-X cycle DPm validation from saved data

This notebook reads the saved W7-X cycle data and validates the renamed API conventions:
- `DX_pol(phi_e)` is the full-period tangent matrix of the continuous-time flow.
- `DPm` is the Jacobian of the full-period Poincare map.
- along the ring, the commutator-evolved `DPm(phi)` is initialized from `DX_pol(phi_e)`.

No original W7-X raw data are embedded into this notebook. The notebook only reads the saved `.h5` file.


In [ ]:
from pathlib import Path
import json
import numpy as np
import h5py

OUTDIR = Path(r'D:/MCFdata/tutorials/W7X_monodromy')
OUTDIR.mkdir(parents=True, exist_ok=True)
DATA = Path(r'D:/MCFdata/w7x/w7x_standard_cycle1.h5')
DATA


In [ ]:
with h5py.File(DATA, 'r') as f:
    phi_DPm = f['cyc/t_DPm'][:]
    DPm_ring = np.transpose(f['cyc/DPm'][:], (2, 0, 1))
    phi_DX_pol = f['cyc/t_Jac'][:]
    DX_pol = np.transpose(f['cyc/Jac'][:], (2, 0, 1))
    X_pol = np.transpose(f['cyc/Xpol'][:], (1, 0)) if f['cyc/Xpol'].shape[0] == 2 else f['cyc/Xpol'][:]

DPm = DX_pol[-1]
summary = {
    'phi_start': float(phi_DX_pol[0]),
    'phi_e': float(phi_DX_pol[-1]),
    'n_DX_pol': int(len(phi_DX_pol)),
    'n_DPm_ring': int(len(phi_DPm)),
    'DPm_from_DX_pol_end': DPm.tolist(),
    'DPm_ring_at_phi0': DPm_ring[0].tolist(),
    'maxabs_DPm_ring_phi0_minus_DPm': float(np.max(np.abs(DPm_ring[0] - DPm))),
    'maxabs_DPm_ring_periodicity_error': float(np.max(np.abs(DPm_ring[-1] - DPm_ring[0]))),
    'eigvals_DPm': np.linalg.eigvals(DPm).tolist(),
    'det_DPm': float(np.linalg.det(DPm)),
}

(OUTDIR / 'w7x_cycle1_DPm_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
summary


In [ ]:
assert np.max(np.abs(DPm_ring[0] - DPm)) < 1e-12
assert abs(np.linalg.det(DPm) - 1.0) < 1e-1
np.savez(OUTDIR / 'w7x_cycle1_DPm_arrays.npz',
         phi_DPm=phi_DPm,
         DPm_ring=DPm_ring,
         phi_DX_pol=phi_DX_pol,
         DX_pol=DX_pol)
'saved'
